### 安裝套件

In [12]:
!pip install requests beautifulsoup4 selenium

In [13]:
import requests
from bs4 import BeautifulSoup

def fetch_page_with_redirect_check(url: str, timeout: int = 10):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        # 關鍵點：allow_redirects=False 會禁止自動跳轉，可抓到 301/302 狀態
        resp = requests.get(url, headers=headers, timeout=timeout, allow_redirects=False)
        print(f"[Debug] 存取 URL: {url}")
        print(f"[Debug] 伺服器回傳狀態碼: {resp.status_code}")

        if resp.status_code in (301, 302):
            # 從 Header 中的 'Location' 抓出目的地
            redirect_url = resp.headers.get('Location')
            print(f"!!! 偵測到重導向 ({resp.status_code}) !!!")
            print(f"請登入後再存取，或檢查 Cookie。")
            print(f"跳轉目的地: {redirect_url}")
            return None
        
        elif resp.status_code == 200:
            print("成功取得頁面內容（200 OK）")
            # 即使是 200，也要檢查內容是不是偽裝的登入頁
            if "login" in resp.url or "尚未登入" in resp.text:
                 print("警告：雖然狀態碼是 200，但內容似乎是登入頁面。")
            return resp.text
        
        else:
            print(f"收到了非預期的狀態碼: {resp.status_code}")
            return None

    except requests.RequestException as e:
        print(f"[ERROR] 請求發生異常: {e}")
        return None

def main():
    # 這是需要權限的 Moodle 討論區網址
    url = "https://moodle3.ntnu.edu.tw/mod/forum/discuss.php?d=491764"
    
    html = fetch_page_with_redirect_check(url)
    
    if html:
        print("\n--- 成功抓取到 HTML 內容 (前 100 字) ---")
        print(html[:100].strip())
    else:
        print("\n--- 無法解析內容：未通過身分驗證或頁面不存在 ---")

if __name__ == "__main__":
    main()

[Debug] 存取 URL: https://moodle3.ntnu.edu.tw/mod/forum/discuss.php?d=491764
[Debug] 伺服器回傳狀態碼: 303
收到了非預期的狀態碼: 303

--- 無法解析內容：未通過身分驗證或頁面不存在 ---


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_moodle_with_selenium(target_url, username, password):
    # 1. 設定 Chrome Options
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # 若穩定後可開啟無頭模式 (不顯示視窗)
    chrome_options.add_argument("--disable-notifications")

    # 啟動 WebDriver (請確保已安裝與 Chrome 版本對應的 chromedriver)
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        # 2. 前往 Moodle 登入頁面
        print("正在前往登入頁面...")
        driver.get("https://moodle3.ntnu.edu.tw/login/index.php")

        # 3. 等待輸入框出現並填入帳密
        wait = WebDriverWait(driver, 10)
        user_input = wait.until(EC.presence_of_element_located((By.ID, "username")))
        pass_input = driver.find_element(By.ID, "password")

        user_input.send_keys(username)
        pass_input.send_keys(password)

        # 4. 點擊登入按鈕
        login_btn = driver.find_element(By.ID, "loginbtn")
        login_btn.click()

        # 5. 等待跳轉回主頁 (檢查是否有登出按鈕或個人頭像來判斷成功)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "usermenu")))
        print("登入成功！")

        # 6. 前往目標討論區網址
        print(f"正在前往目標頁面: {target_url}")
        driver.get(target_url)

        # 7. 等待討論區內容載入
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "discussionname")))

        # 8. 抓取 HTML 並交給 BeautifulSoup (或直接用 Selenium 抓)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        title = soup.select_one("h3.discussionname").get_text(strip=True)
        print(f"\n文章標題: {title}")

        posts = soup.select(".forumpost")
        print(f"共抓取到 {len(posts)} 則回覆。")
        
        for i, post in enumerate(posts, 1):
            print(f"[{i}] {post.get_text(strip=True)[:50]}...")

    except Exception as e:
        print(f"發生錯誤: {e}")
    
    finally:
        # 觀察完畢後關閉瀏覽器
        time.sleep(5)
        #driver.quit()

if __name__ == "__main__":
    from bs4 import BeautifulSoup
    
    # 請填入你的 NTNU 帳密
    USER = "41171112H"
    PASSWORD = ""
    TARGET = "https://moodle3.ntnu.edu.tw/mod/forum/discuss.php?d=501110"

    scrape_moodle_with_selenium(TARGET, USER, PASSWORD)

正在前往登入頁面...
登入成功！
正在前往目標頁面: https://moodle3.ntnu.edu.tw/mod/forum/discuss.php?d=501110

文章標題: HW3
共抓取到 25 則回覆。
[1] HW3由pecu 蔡芸琤發表於2026年 04月 23日(四) 10:06回覆數： 24請分享你是如...
[2] 回覆 pecu 蔡芸琤回應: HW3由41371208H 吳以謙發表於2026年 05月 7日(四)...
[3] 回覆 pecu 蔡芸琤回應: HW3由41371218H 賴建辰發表於2026年 05月 11日(一...
[4] 回覆 pecu 蔡芸琤回應: HW3由41371209H 林秉翰發表於2026年 05月 12日(二...
[5] 回覆 pecu 蔡芸琤回應: HW3由41371213H 秦子琋發表於2026年 05月 12日(二...
[6] 回覆 pecu 蔡芸琤回應: HW3由41371223H 洪于涵發表於2026年 05月 12日(二...
[7] 回覆 pecu 蔡芸琤回應: HW3由41371204H 李雅郁發表於2026年 05月 13日(三...
[8] 回覆 pecu 蔡芸琤回應: HW3由41371211H 周怡辰發表於2026年 05月 13日(三...
[9] 回覆 pecu 蔡芸琤回應: HW3由41371229H 范子昊發表於2026年 05月 13日(三...
[10] 回覆 pecu 蔡芸琤回應: HW3由41371210H 劉祐羽發表於2026年 05月 13日(三...
[11] 回覆 pecu 蔡芸琤回應: HW3由41371221H 賴柏妤發表於2026年 05月 13日(三...
[12] 回覆 pecu 蔡芸琤回應: HW3由41340111S 王羿發表於2026年 05月 13日(三)...
[13] 回覆 pecu 蔡芸琤回應: HW3由41371216H 楊立宇發表於2026年 05月 13日(三...
[14] 回覆 pecu 蔡芸琤回應: HW3由41040313S 丁祺發表於2026年 05月 13日(三)...
[15] 回覆 pecu 蔡芸琤回應: HW3由61271012H 張浩文發表於2026年 05月 14日(四...
[16] 回覆 pecu 